# 02 · Financial Ratio Analysis in SQL

Loads the cleaned statements into a SQLite database and computes profitability, efficiency, liquidity and cash-flow metrics in SQL, then adjusts for one-time items.

**Techniques:** schema design with primary keys · idempotent loads · `CASE WHEN` pivot · views · window functions (`LAG`, moving averages) · CTEs · `LEFT JOIN` adjustments

## Setup

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

BASE = Path.cwd()
CLEAN_DIR = BASE / "data" / "clean"
OUT_DIR = BASE / "data" / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = BASE / "data" / "formfactor.db"

conn = sqlite3.connect(DB_PATH)
def q(sql):
    return pd.read_sql_query(sql, conn)

## 1. Schema

`financials` is a long fact table keyed on `(ticker, fiscal_year, metric)`. The primary key makes duplicate loads fail loudly instead of silently double-counting, and the `ticker` column lets the same table hold peer companies later (notebook 03).

In [2]:
_ = conn.executescript("""
CREATE TABLE IF NOT EXISTS financials (
    ticker       TEXT    NOT NULL,
    fiscal_year  INTEGER NOT NULL,
    statement    TEXT    NOT NULL,   -- IS / BS / CF
    metric       TEXT    NOT NULL,
    value        REAL,
    concept      TEXT,               -- source XBRL tag, or 'derived'
    period_end   TEXT,
    filed_date   TEXT,
    accn         TEXT,
    PRIMARY KEY (ticker, fiscal_year, metric)
);
CREATE TABLE IF NOT EXISTS metric_dictionary (
    metric TEXT PRIMARY KEY, statement TEXT, xbrl_candidates TEXT
);
""")

## 2. Load (idempotent: delete then insert, so re-running never duplicates rows)

In [3]:
fin = pd.read_csv(CLEAN_DIR / "formfactor_financials_long.csv")
dic = pd.read_csv(CLEAN_DIR / "metric_dictionary.csv")

conn.execute("DELETE FROM financials WHERE ticker = 'FORM'")
conn.execute("DELETE FROM metric_dictionary")
fin.to_sql("financials", conn, if_exists="append", index=False)
dic.to_sql("metric_dictionary", conn, if_exists="append", index=False)
conn.commit()

q("SELECT COUNT(*) AS rows, COUNT(DISTINCT metric) AS metrics, MIN(fiscal_year) AS first_fy, MAX(fiscal_year) AS last_fy FROM financials")

,rows,metrics,first_fy,last_fy
0,278,28,2016,2025


## 3. Revenue trend

In [4]:
q("""
SELECT fiscal_year, ROUND(value / 1e6, 1) AS revenue_musd
FROM financials
WHERE metric = 'revenue'
ORDER BY fiscal_year
""")

,fiscal_year,revenue_musd
0,2016,383.90
1,2017,548.40
2,2018,529.70
3,2019,589.50
4,2020,693.60
5,2021,769.70
6,2022,747.90
7,2023,663.10
8,2024,763.60
9,2025,785.00


## 4. Pivot long → wide as a view

`MAX(CASE WHEN metric = 'x' THEN value END)` turns one row per metric into one column per metric. The SQL for all 28 metrics is generated in Python and stored as the view `fin_wide`.

In [5]:
metrics = q("SELECT DISTINCT metric FROM financials ORDER BY metric")["metric"].tolist()
case_lines = ",\n       ".join(f"MAX(CASE WHEN metric = '{m}' THEN value END) AS {m}" for m in metrics)

conn.execute("DROP VIEW IF EXISTS fin_wide")
conn.execute(f"""
CREATE VIEW fin_wide AS
SELECT ticker, fiscal_year,
       {case_lines}
FROM financials
GROUP BY ticker, fiscal_year
""")
q("SELECT fiscal_year, revenue, gross_profit, net_income, total_equity FROM fin_wide WHERE ticker = 'FORM'").set_index("fiscal_year").div(1e6).round(1)

,revenue,gross_profit,net_income,total_equity
fiscal_year,,,,
2016,383.90,102.70,-6.60,401.10
2017,548.40,215.60,40.90,458.60
2018,529.70,210.30,104.00,580.20
2019,589.50,237.50,39.30,641.00
2020,693.60,287.90,78.50,744.10
2021,769.70,322.80,83.90,815.80
2022,747.90,296.00,50.70,808.30
2023,663.10,258.60,82.40,908.80
2024,763.60,307.90,69.60,947.80


## 5. Financial ratios

| Group | Metrics |
|---|---|
| Profitability | gross, operating and net margin |
| Investment | R&D and SG&A as % of revenue, capex intensity |
| Efficiency | ROA (ROE in section 6) |
| Liquidity & leverage | current ratio, debt-to-equity, net cash |
| Cash generation | free cash flow and FCF margin |
| Quality | stock-based compensation as % of revenue |

`NULLIF(x, 0)` guards against division by zero; `COALESCE(x, 0)` treats a missing short-term investment balance as zero.

In [6]:
conn.execute("DROP VIEW IF EXISTS ratios")
conn.execute("""
CREATE VIEW ratios AS
SELECT
    ticker, fiscal_year,
    revenue / 1e6                                                  AS revenue_musd,
    100.0 * gross_profit     / NULLIF(revenue, 0)                  AS gross_margin_pct,
    100.0 * operating_income / NULLIF(revenue, 0)                  AS operating_margin_pct,
    100.0 * net_income       / NULLIF(revenue, 0)                  AS net_margin_pct,
    100.0 * rd_expense       / NULLIF(revenue, 0)                  AS rd_pct_revenue,
    100.0 * sga_expense      / NULLIF(revenue, 0)                  AS sga_pct_revenue,
    100.0 * net_income       / NULLIF(total_assets, 0)             AS roa_pct,
    current_assets           / NULLIF(current_liabilities, 0)      AS current_ratio,
    total_liabilities        / NULLIF(total_equity, 0)             AS debt_to_equity,
    (operating_cash_flow - capex) / 1e6                            AS fcf_musd,
    100.0 * (operating_cash_flow - capex) / NULLIF(revenue, 0)     AS fcf_margin_pct,
    (cash + COALESCE(st_investments, 0) - COALESCE(long_term_debt, 0)) / 1e6 AS net_cash_musd,
    100.0 * capex            / NULLIF(revenue, 0)                  AS capex_pct_revenue,
    100.0 * stock_comp       / NULLIF(revenue, 0)                  AS sbc_pct_revenue
FROM fin_wide
""")
q("SELECT * FROM ratios WHERE ticker = 'FORM' ORDER BY fiscal_year").drop(columns="ticker").set_index("fiscal_year").round(1)

,revenue_musd,gross_margin_pct,operating_margin_pct,net_margin_pct,rd_pct_revenue,sga_pct_revenue,roa_pct,current_ratio,debt_to_equity,fcf_musd,fcf_margin_pct,net_cash_musd,capex_pct_revenue,sbc_pct_revenue
fiscal_year,,,,,,,,,,,,,,
2016,383.90,26.70,-12.50,-1.70,15.00,19.10,-1.10,3.10,0.50,NaN,NaN,-24.10,3.00,3.00
2017,548.40,39.30,8.40,7.50,13.50,17.40,6.30,3.30,0.40,68.60,12.50,52.90,3.20,3.00
2018,529.70,39.70,6.80,19.60,14.20,18.70,14.30,3.30,0.30,48.80,9.20,114.00,3.80,3.40
2019,589.50,40.30,8.40,6.70,13.80,18.00,4.70,3.10,0.30,100.20,17.00,205.20,3.50,3.90
2020,693.60,41.50,12.10,11.30,12.80,16.60,8.20,3.20,0.30,113.40,16.30,230.10,8.10,3.40
2021,769.70,41.90,12.70,10.90,13.10,16.10,8.20,3.50,0.30,72.90,9.50,260.60,8.60,3.80
2022,747.90,39.60,7.30,6.80,14.60,17.60,5.00,3.20,0.20,66.50,8.90,223.70,8.70,4.20
2023,663.10,39.00,12.50,12.40,17.50,20.10,7.40,4.40,0.20,8.60,1.30,315.00,8.40,5.80
2024,763.60,40.30,8.50,9.10,16.00,18.60,6.10,4.60,0.20,79.10,10.40,347.80,5.00,5.20


## 6. Growth and returns with window functions

`LAG(x) OVER w` reads the prior year's value. ROE uses average equity (opening = prior year-end). `PARTITION BY ticker` keeps companies separate once peers are added.

In [7]:
q("""
SELECT fiscal_year,
       ROUND(revenue / 1e6, 1)                                                          AS revenue_musd,
       ROUND(100.0 * (revenue - LAG(revenue) OVER w) / LAG(revenue) OVER w, 1)          AS revenue_yoy_pct,
       ROUND(100.0 * (net_income - LAG(net_income) OVER w) / ABS(LAG(net_income) OVER w), 1) AS net_income_yoy_pct,
       ROUND(100.0 * (rd_expense - LAG(rd_expense) OVER w) / LAG(rd_expense) OVER w, 1) AS rd_yoy_pct,
       ROUND(100.0 * net_income / ((total_equity + LAG(total_equity) OVER w) / 2.0), 1) AS roe_pct
FROM fin_wide
WHERE ticker = 'FORM'
WINDOW w AS (PARTITION BY ticker ORDER BY fiscal_year)
ORDER BY fiscal_year
""")

,fiscal_year,revenue_musd,revenue_yoy_pct,net_income_yoy_pct,rd_yoy_pct,roe_pct
0,2016,383.90,NaN,NaN,NaN,NaN
1,2017,548.40,42.90,724.00,28.50,9.50
2,2018,529.70,-3.40,154.30,1.60,20.00
3,2019,589.50,11.30,-62.20,8.70,6.40
4,2020,693.60,17.70,99.60,9.20,11.30
5,2021,769.70,11.00,6.90,13.40,10.80
6,2022,747.90,-2.80,-39.50,8.20,6.20
7,2023,663.10,-11.30,62.40,6.00,9.60
8,2024,763.60,15.20,-15.50,5.30,7.50
9,2025,785.00,2.80,-21.90,-5.10,5.50


## 7. One-time items

FY2023 revenue fell 11.3% while GAAP net income rose 62%. Reconciling reported operating income against `gross profit − R&D − SG&A` isolates the gap.

In [8]:
q("""
SELECT fiscal_year,
       ROUND(operating_income / 1e6, 1)                                               AS reported_op_income,
       ROUND((gross_profit - rd_expense - sga_expense) / 1e6, 1)                      AS gp_less_rd_sga,
       ROUND((operating_income - (gross_profit - rd_expense - sga_expense)) / 1e6, 1) AS other_operating_items
FROM fin_wide
WHERE ticker = 'FORM' AND fiscal_year >= 2020
ORDER BY fiscal_year
""")

,fiscal_year,reported_op_income,gp_less_rd_sga,other_operating_items
0,2020,83.80,83.80,0.00
1,2021,98.00,98.00,0.00
2,2022,54.90,54.90,0.00
3,2023,82.80,9.80,73.00
4,2024,64.80,44.20,20.60
5,2025,57.10,60.10,-3.00


The $73.0M gap in FY2023 matches the **gain on the sale of the FRT business** disclosed in the Q4 2023 earnings release. One-time items are stored in their own table, with a source, and subtracted at query time; the reported data is never edited.

In [9]:
_ = conn.executescript("""
CREATE TABLE IF NOT EXISTS one_time_items (
    ticker TEXT, fiscal_year INTEGER, item TEXT, amount REAL, source TEXT,
    PRIMARY KEY (ticker, fiscal_year, item)
);
DELETE FROM one_time_items WHERE ticker = 'FORM';
INSERT INTO one_time_items VALUES
    ('FORM', 2023, 'Gain on sale of FRT business', 73.0e6, 'FormFactor Q4 2023 earnings release');
""")
conn.commit()

q("""
WITH adj AS (
    SELECT ticker, fiscal_year, SUM(amount) AS one_time FROM one_time_items GROUP BY ticker, fiscal_year
),
base AS (
    SELECT f.fiscal_year,
           100.0 * f.operating_income / f.revenue                              AS opm_reported,
           100.0 * (f.operating_income - COALESCE(a.one_time, 0)) / f.revenue AS opm_adjusted
    FROM fin_wide f
    LEFT JOIN adj a ON f.ticker = a.ticker AND f.fiscal_year = a.fiscal_year
    WHERE f.ticker = 'FORM' AND f.fiscal_year >= 2017
)
SELECT fiscal_year,
       ROUND(opm_reported, 1)             AS opm_reported,
       ROUND(AVG(opm_reported) OVER w, 1) AS opm_3yr_reported,
       ROUND(opm_adjusted, 1)             AS opm_adjusted,
       ROUND(AVG(opm_adjusted) OVER w, 1) AS opm_3yr_adjusted
FROM base
WINDOW w AS (ORDER BY fiscal_year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
ORDER BY fiscal_year
""")

,fiscal_year,opm_reported,opm_3yr_reported,opm_adjusted,opm_3yr_adjusted
0,2017,8.40,8.40,8.40,8.40
1,2018,6.80,7.60,6.80,7.60
2,2019,8.40,7.90,8.40,7.90
3,2020,12.10,9.10,12.10,9.10
4,2021,12.70,11.10,12.70,11.10
5,2022,7.30,10.70,7.30,10.70
6,2023,12.50,10.90,1.50,7.20
7,2024,8.50,9.40,8.50,5.80
8,2025,7.30,9.40,7.30,5.70


FY2018 shows a similar distortion below the operating line: net income of $104M against operating income of $36M, driven by a roughly $70M income-tax benefit (`income_tax` = −$70.1M).

## 8. Long-run growth (CAGR, 2017 → latest)

SQLite has no `POWER()`, so SQL retrieves the endpoints and Python computes the compound rate.

In [10]:
ends = q("""
WITH s AS (SELECT * FROM fin_wide WHERE ticker = 'FORM' AND fiscal_year = 2017),
     e AS (SELECT * FROM fin_wide WHERE ticker = 'FORM' AND fiscal_year = (SELECT MAX(fiscal_year) FROM fin_wide WHERE ticker = 'FORM'))
SELECT s.fiscal_year AS y0, e.fiscal_year AS y1,
       s.revenue AS rev0, e.revenue AS rev1, s.rd_expense AS rd0, e.rd_expense AS rd1,
       s.net_income AS ni0, e.net_income AS ni1, s.total_equity AS eq0, e.total_equity AS eq1
FROM s CROSS JOIN e
""").iloc[0]

n = ends.y1 - ends.y0
cagr = pd.Series({
    "Revenue": (ends.rev1 / ends.rev0) ** (1 / n) - 1,
    "R&D expense": (ends.rd1 / ends.rd0) ** (1 / n) - 1,
    "Net income": (ends.ni1 / ends.ni0) ** (1 / n) - 1,
    "Stockholders' equity": (ends.eq1 / ends.eq0) ** (1 / n) - 1,
}).mul(100).round(1)
cagr.to_frame(f"CAGR % {int(ends.y0)}–{int(ends.y1)}")

,CAGR % 2017–2025
Revenue,4.60
R&D expense,5.80
Net income,3.60
Stockholders' equity,10.70


## 9. Performance through the cycle

In [11]:
q("""
WITH adj AS (SELECT ticker, fiscal_year, SUM(amount) AS one_time FROM one_time_items GROUP BY ticker, fiscal_year),
tagged AS (
    SELECT r.*,
           100.0 * (f.operating_income - COALESCE(a.one_time, 0)) / f.revenue AS adj_opm,
           CASE WHEN r.fiscal_year BETWEEN 2017 AND 2019 THEN '1. 2017-19 pre-pandemic'
                WHEN r.fiscal_year BETWEEN 2020 AND 2022 THEN '2. 2020-22 chip upcycle'
                ELSE                                         '3. 2023-25 downturn & AI recovery' END AS phase
    FROM ratios r
    JOIN fin_wide f ON r.ticker = f.ticker AND r.fiscal_year = f.fiscal_year
    LEFT JOIN adj a ON r.ticker = a.ticker AND r.fiscal_year = a.fiscal_year
    WHERE r.ticker = 'FORM' AND r.fiscal_year >= 2017
)
SELECT phase,
       ROUND(AVG(revenue_musd), 0)       AS avg_revenue_musd,
       ROUND(AVG(gross_margin_pct), 1)   AS avg_gross_margin,
       ROUND(AVG(adj_opm), 1)            AS avg_op_margin_adj,
       ROUND(AVG(rd_pct_revenue), 1)     AS avg_rd_pct,
       ROUND(AVG(capex_pct_revenue), 1)  AS avg_capex_pct,
       ROUND(SUM(fcf_musd), 0)           AS total_fcf_musd
FROM tagged
GROUP BY phase
ORDER BY phase
""")

,phase,avg_revenue_musd,avg_gross_margin,avg_op_margin_adj,avg_rd_pct,avg_capex_pct,total_fcf_musd
0,1. 2017-19 pre-pandemic,556.00,39.80,7.90,13.80,3.50,218.00
1,2. 2020-22 chip upcycle,737.00,41.00,10.70,13.50,8.50,253.00
2,3. 2023-25 downturn & AI recovery,737.00,39.60,5.70,16.10,8.90,99.00


## 10. Capital allocation

In [12]:
q("""
SELECT fiscal_year,
       ROUND(operating_cash_flow / 1e6, 1) AS ocf,
       ROUND(capex / 1e6, 1)               AS capex,
       ROUND(acquisitions / 1e6, 1)        AS acquisitions,
       ROUND(buybacks / 1e6, 1)            AS buybacks,
       ROUND(stock_comp / 1e6, 1)          AS stock_comp,
       ROUND(shares_diluted / 1e6, 1)      AS diluted_shares_m,
       ROUND(100.0 * (capex + acquisitions + buybacks) / NULLIF(operating_cash_flow, 0), 0) AS pct_of_ocf_deployed
FROM fin_wide
WHERE ticker = 'FORM' AND fiscal_year >= 2017
ORDER BY fiscal_year
""")

,fiscal_year,ocf,capex,acquisitions,buybacks,stock_comp,diluted_shares_m,pct_of_ocf_deployed
0,2017,86.30,17.80,0.00,19.00,16.30,74.20,43.00
1,2018,68.70,19.90,0.00,0.00,17.80,75.20,29.00
2,2019,121.00,20.80,20.50,0.00,23.20,77.30,34.00
3,2020,169.30,55.90,51.90,0.00,23.80,79.00,64.00
4,2021,139.40,66.50,0.00,24.00,29.40,79.10,65.00
5,2022,131.80,65.30,3.40,82.30,31.30,78.20,115.00
6,2023,64.60,56.00,0.00,19.80,38.60,78.20,117.00
7,2024,117.50,38.40,0.00,53.30,39.80,78.40,78.00
8,2025,115.40,103.70,20.60,26.20,38.60,78.30,130.00


## Export

In [13]:
final = q("""
SELECT r.*,
       100.0 * (r.revenue_musd - LAG(r.revenue_musd) OVER w) / LAG(r.revenue_musd) OVER w AS revenue_yoy_pct,
       100.0 * f.net_income / ((f.total_equity + LAG(f.total_equity) OVER w) / 2.0)        AS roe_pct
FROM ratios r
JOIN fin_wide f ON r.ticker = f.ticker AND r.fiscal_year = f.fiscal_year
WHERE r.ticker = 'FORM'
WINDOW w AS (PARTITION BY r.ticker ORDER BY r.fiscal_year)
ORDER BY r.fiscal_year
""")
final.round(2).to_csv(OUT_DIR / "formfactor_ratios.csv", index=False)
conn.close()

## Findings

- **Slow but steady growth.** Revenue compounded at 4.6% a year from 2017 to 2025 ($548M → $785M), with declines in 2022 (−2.8%) and 2023 (−11.3%).
- **Stable gross margin, weaker operating margin.** Gross margin stayed between 39% and 42%. Excluding the FY2023 FRT gain, the three-year average operating margin fell from 11.1% (2021) to 5.7% (2025): R&D and SG&A grew faster than revenue.
- **Reported earnings need adjusting.** FY2023 operating income of $82.8M includes a $73.0M divestiture gain (underlying ≈ $9.8M, a 1.5% margin); FY2018 net income includes a ~$70M tax benefit.
- **Cash generation weakened.** Three-year free cash flow fell from $253M (2020–22) to $99M (2023–25) on flat average revenue, mainly because operating cash flow dropped $143M. Capex hit a record $104M in 2025, 13% of revenue.
- **Conservative balance sheet, modest returns.** Debt-to-equity is about 0.2 and net cash about $264M, but ROE has ranged from 5% to 11% since 2019. Buybacks largely offset stock-based compensation; diluted shares have held near 78M since 2020.